In [5]:
# !pip install pandas
!pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]


In [12]:
import pandas as pd
import math
import pickle
from pathlib import Path
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# ———————– CONFIG ———————————————————————————————
EXCEL_PATH = Path("Dealer address.xlsx")
CACHE_PATH = Path("/mnt/data/dealer_geo_cache.pkl")
USER_AGENT = "hrj-dealer-finder"

# ———————– UTILITIES ———————————————————————————————
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    φ1, φ2 = map(math.radians, (lat1, lat2))
    dφ = math.radians(lat2 - lat1)
    dλ = math.radians(lon2 - lon1)
    a = math.sin(dφ/2)**2 + math.cos(φ1)*math.cos(φ2)*math.sin(dλ/2)**2
    return 2 * R * math.asin(math.sqrt(a))

geocoder = Nominatim(user_agent=USER_AGENT)
geocode = RateLimiter(geocoder.geocode, min_delay_seconds=1, max_retries=2)

def load_dealers():
    df = pd.read_excel(EXCEL_PATH)
    # Build a full address string for geocoding
    df["full_address"] = (
        df["Address"].astype(str)
        + ", "
        + df["Postal Code"].astype(str)
        + ", India"
    )
    return df

def cache_geocodes(df):
    if CACHE_PATH.exists():
        geo_cache = pickle.loads(CACHE_PATH.read_bytes())
    else:
        geo_cache = {}

    for addr in df["full_address"].unique():
        if addr not in geo_cache:
            loc = geocode(addr, timeout=10)
            geo_cache[addr] = (loc.latitude, loc.longitude) if loc else (None, None)

    CACHE_PATH.write_bytes(pickle.dumps(geo_cache))
    return geo_cache

def find_nearest_retailer(user_pin: str):
    # 1. Geocode the user’s PIN
    user_loc = geocode(f"{user_pin}, India", timeout=10)
    if not user_loc:
        raise ValueError(f"Could not geolocate PIN {user_pin}")

    # 2. Load & cache dealer geos
    df = load_dealers()
    geo_cache = cache_geocodes(df)

    # 3. Map cached coords
    df["lat"] = df["full_address"].map(lambda a: geo_cache[a][0])
    df["lon"] = df["full_address"].map(lambda a: geo_cache[a][1])
    df = df.dropna(subset=["lat", "lon"])

    # 4. Compute distances
    df["distance_km"] = df.apply(
        lambda r: haversine(
            user_loc.latitude, user_loc.longitude, r["lat"], r["lon"]
        ),
        axis=1,
    )

    # 5. Select nearest
    best = df.nsmallest(1, "distance_km").iloc[0].to_dict()
    best["distance_km"] = round(best["distance_km"], 1)
    return {
        "dealer_name": best["Name"],
        "address": best["Address"],
        "postal_code": best["Postal Code"],
        "distance_km": best["distance_km"],
        # you can add other fields like Tel, Region, etc.
    }



In [13]:
# ———————– USAGE ———————————————————————————————
if __name__ == "__main__":
    pin = "226016"
    nearest = find_nearest_retailer(pin)
    print(f"Nearest dealer to {pin} is {nearest['dealer_name']} at {nearest['distance_km']} km")


KeyboardInterrupt: 

In [14]:
"""
dealer_lookup.py  ──  Excel-powered dealer directory
====================================================

Functions
---------
load_dealers()          → DataFrame with standardised columns
states()                → list of all distinct states
cities_with_dealers()   → list of cities that have at least one dealer in a state
dealers_in()            → list[dict] of dealers for a given (city, state)
"""

from pathlib import Path
from functools import lru_cache
from typing import List, Dict

import pandas as pd

# ── CONFIG ────────────────────────────────────────────────────────────────
EXCEL_PATH = Path("./Dealer address.xlsx")  # change if needed

# ── CORE LOADER ───────────────────────────────────────────────────────────


@lru_cache(maxsize=1)
def load_dealers() -> pd.DataFrame:
    """
    Read the Excel sheet once, standardise column names,
    and strip whitespace for safer matching.
    """
    df = pd.read_excel(EXCEL_PATH)

    # Rename to snake_case we’ll use everywhere else
    df = df.rename(
        columns={
            "Region": "state",
            "City": "city",
            "Name": "dealer_name",
            "Address": "address",
            "Postal Code": "postal_code",
            "Tel": "phone",
        }
    )

    # Normalise text columns
    for col in ("state", "city", "dealer_name"):
        df[col] = df[col].astype(str).str.strip()

    return df


# ── PUBLIC API ────────────────────────────────────────────────────────────


def states() -> List[str]:
    """
    Return all distinct states (sorted).
    """
    return sorted(load_dealers()["state"].unique())


def cities_with_dealers(state: str) -> List[str]:
    """
    Return a sorted list of cities that host at least one dealer
    in the requested *state* (case-insensitive).
    """
    df = load_dealers()
    mask = df["state"].str.casefold() == state.casefold()
    return sorted(df.loc[mask, "city"].unique())


def dealers_in(city: str, state: str) -> List[Dict]:
    """
    Return dealer rows for the given (city, state) as a list of dicts.
    Matching is case-insensitive.  If nothing matches, returns [].
    Dicts include: dealer_name, address, postal_code, phone.
    """
    df = load_dealers()
    mask = (
        (df["state"].str.casefold() == state.casefold())
        & (df["city"].str.casefold() == city.casefold())
    )
    cols = ["dealer_name", "address", "postal_code", "phone"]
    return df.loc[mask, cols].to_dict(orient="records")




In [19]:
# ── DEMO USAGE ────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("States available:", states())
    up_cities = cities_with_dealers("UP")
    print(f"Cities with dealers in Uttar Pradesh ({len(up_cities)}):", up_cities)

    lucknow_dealers = dealers_in("Lucknow", "UP")
    print(f"Dealers in Lucknow, UP ({len(lucknow_dealers)}):")
    for d in lucknow_dealers:
        print(" •", d["dealer_name"], "—", d["address"], "-", d["phone"])



States available: ['AN', 'AP', 'ARP', 'AS', 'BH', 'CD', 'CG', 'DEL', 'GDD', 'GJ', 'HP', 'HR', 'JH', 'JK', 'KAR', 'KER', 'MAH', 'MG', 'MP', 'NG', 'NP', 'OR', 'PB', 'PY', 'RJ', 'SK', 'TG', 'TN', 'TR', 'UP', 'UT', 'WB']
Cities with dealers in Uttar Pradesh (48): ['AGRA', 'AURAIYA', 'Agra', 'Allahabad', 'Ambedkar Nagar', 'Ayodha', 'Azamgarh', 'BAHRAICH,', 'BALLIA', 'BARABANKI', 'BIJNOR', 'Ballia', 'Bulandshahr', 'Distt.- Ambedkar Nagar', 'Gautam Buddha Nagar', 'Ghaziabad', 'Ghazipur', 'HAMIRPUR', 'JHANSI', 'Jalaun', 'Jhansi', 'KANPUR', 'KAUSHAMBI', 'Kanpur Nagar', 'Kheri', 'Kushinagar', 'LAKHIMPUR  KHERI', 'LALITPUR', 'LUCKNOW', 'Lucknow', 'MEERUT', 'Mahrajganj', 'Mathura', 'Mau', 'Meerut', 'Moradabad', 'Muzaffarnagar', 'Noida', 'ORAI', 'SAHARANPUR,', 'SITAPUR', 'Saharanpur', 'Sant Ravidas Nagar (Bhadohi)', 'Siddharthnagar', 'VARANASI', 'VARANASI,', 'Varanasi', 'ghazipur']
Dealers in Lucknow, UP (7):
 • A K ENTERPRISES — HIG.2 JAWAHAR VIHAR COLONY, SULTANPUR ROAD ,RAEBARELI LUCKNOW - 94157

In [ ]:
from pathlib import Path
from functools import lru_cache
from typing import List, Dict

import pandas as pd

# ── CONFIG ────────────────────────────────────────────────────────────────
# Robustly locate the Excel file
try:                                   # running as a .py script
    BASE_DIR = Path(__file__).parent
except NameError:                      # running in a notebook / REPL
    BASE_DIR = Path.cwd()

EXCEL_PATH = BASE_DIR / "Dealer address.xlsx"   # adjust filename if needed



# ── STATE CODE ↔ NAME MAP (extend as needed) ──────────────────────────────
_STATE_CODE_MAP = {
    # 2–3-letter codes → canonical full names
    "AN":  "Andaman & Nicobar Islands",
    "AP":  "Andhra Pradesh",
    "ARP": "Arunachal Pradesh",
    "AS":  "Assam",
    "BH":  "Bihar",
    "CG":  "Chhattisgarh",
    "CH":  "Chandigarh",
    "DD":  "Daman & Diu",
    "DL":  "Delhi",
    "GA":  "Goa",
    "GJ":  "Gujarat",
    "HP":  "Himachal Pradesh",
    "HR":  "Haryana",
    "JK":  "Jammu & Kashmir",
    "JH":  "Jharkhand",
    "KA":  "Karnataka",
    "KL":  "Kerala",
    "LD":  "Lakshadweep",
    "MH":  "Maharashtra",
    "MP":  "Madhya Pradesh",
    "MN":  "Manipur",
    "ML":  "Meghalaya",
    "MZ":  "Mizoram",
    "NL":  "Nagaland",
    "OD":  "Odisha",
    "PB":  "Punjab",
    "PY":  "Puducherry",
    "RJ":  "Rajasthan",
    "SK":  "Sikkim",
    "TN":  "Tamil Nadu",
    "TS":  "Telangana",
    "TR":  "Tripura",
    "UP":  "Uttar Pradesh",
    "UK":  "Uttarakhand",
    "WB":  "West Bengal",
}

# Reverse map: full name (casefolded) → code
_NAME_TO_CODE = {name.casefold(): code for code, name in _STATE_CODE_MAP.items()}

# ── INTERNAL HELPERS ──────────────────────────────────────────────────────
def _normalise_state(user_input: str) -> str:
    """
    Accept either a state *code* ("UP") or full name ("Uttar Pradesh")
    and return the canonical code used in the Excel sheet.
    """
    s = user_input.strip()
    code = s.upper()
    if code in _STATE_CODE_MAP:           # already a code
        return code
    key = s.casefold()
    if key in _NAME_TO_CODE:              # full name given
        return _NAME_TO_CODE[key]
    raise ValueError(
        f"Unknown state '{user_input}'. "
        f"Accepted codes: {', '.join(sorted(_STATE_CODE_MAP))}"
    )

@lru_cache(maxsize=1)
def _load_df() -> pd.DataFrame:
    """
    Read the Excel file once, standardise column names,
    trim whitespace, and cache the DataFrame in memory.
    """
    df = pd.read_excel(EXCEL_PATH)

    df = df.rename(
        columns={
            "Region":      "state",        # code in your sheet
            "City":        "city",
            "Name":        "dealer_name",
            "Address":     "address",
            "Postal Code": "postal_code",
            "Tel":         "phone",
        }
    )

    for col in ("state", "city", "dealer_name"):
        df[col] = df[col].astype(str).str.strip()

    return df

# ── PUBLIC API ────────────────────────────────────────────────────────────
def states() -> List[str]:
    """
    Return all distinct *codes* present in the sheet.
    """
    return sorted(_load_df()["state"].unique())

def cities_with_dealers(state: str) -> List[str]:
    """
    List every city that has ≥1 dealer in the given state.
    *state* may be code ("UP") or full name ("Uttar Pradesh").
    """
    code = _normalise_state(state)
    df   = _load_df()
    return sorted(df.loc[df["state"] == code, "city"].unique())

def dealers_in(city: str, state: str) -> List[Dict]:
    """
    Return dealers for (city, state) as list[dict].
    *state* may be code or full name; city match is case-insensitive.
    """
    code = _normalise_state(state)
    df   = _load_df()
    mask = (df["state"] == code) & (df["city"].str.casefold() == city.casefold())
    cols = ["dealer_name", "address", "postal_code", "phone"]
    return df.loc[mask, cols].to_dict(orient="records")



In [27]:
if __name__ == "__main__":
    print("Dealers in Lucknow, UP:", dealers_in("lucknow", "uttar pradesh")[:3])




Dealers in Lucknow, UP: [{'dealer_name': 'A K ENTERPRISES', 'address': 'HIG.2 JAWAHAR VIHAR COLONY, SULTANPUR ROAD ,RAEBARELI LUCKNOW', 'postal_code': 229010, 'phone': '9415744633'}, {'dealer_name': 'DR.HOME SOLUTIONS PVT LTD', 'address': 'FIRST FLOOR GATA 78,SARTHUA NAGRAM ROAD AMETHI MOHNAN LALGANJ LUCKNOW', 'postal_code': 226002, 'phone': '9891492172'}, {'dealer_name': 'SHREE VISHWANATH TILES AND SANITARY', 'address': '624M/41D MANKHERA TIRAHA,MALHAUR, LUCKNOW', 'postal_code': 226010, 'phone': '9454666607'}]
